# Notebook A: build the DPO data (run once)
**Settings:** Accelerator = None (CPU is enough), Internet = ON. No datasets need to be attached.
When it finishes: Save Version -> Save & Run All (Commit), then on the finished version open **Output -> New Dataset** and name it `dpo-data`.

In [ ]:
import json, random, re
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer

OUT = Path("/kaggle/working/dpo_data")
OUT.mkdir(parents=True, exist_ok=True)
SEED, TRAIN_SIZE, VAL_SIZE = 42, 15_000, 1_000

# Same tokenizer and chat template that SFT and generation used.
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
tok.save_pretrained(OUT / "tokenizer")   # so the DPO notebooks need no internet

def to_prompt(messages):
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print("Tokenizer ready. EOS:", tok.eos_token, "| PAD:", tok.pad_token)

In [ ]:
MARK = "\n\nAssistant:"

def hh_messages(text):
    """'\n\nHuman: ..\n\nAssistant: ..' -> chat messages list."""
    parts = re.split(r"\n\n(Human|Assistant):", text)
    msgs = []
    for role, content in zip(parts[1::2], parts[2::2]):
        content = content.strip()
        if content:
            msgs.append({"role": "user" if role == "Human" else "assistant",
                         "content": content})
    return msgs

def convert_hh(row):
    chosen, rejected = row["chosen"], row["rejected"]
    if not isinstance(chosen, str) or not isinstance(rejected, str):
        return None
    n = 0
    for a, b in zip(chosen, rejected):
        if a != b:
            break
        n += 1
    pos = chosen[:n].rfind(MARK)          # last assistant turn inside the shared prefix
    if pos < 0:
        return None
    end = pos + len(MARK)
    c, r = chosen[end:].strip(), rejected[end:].strip()
    msgs = hh_messages(chosen[:end])
    if not c or not r or c == r or not msgs or msgs[-1]["role"] != "user":
        return None
    if any(t in x for x in (c, r) for t in ("\n\nHuman:", "\n\nAssistant:")):
        return None                        # conversations that differ before the final turn
    return {"prompt": to_prompt(msgs), "chosen": c, "rejected": r}

def last_assistant(messages):
    if isinstance(messages, list):
        for m in reversed(messages):
            if isinstance(m, dict) and m.get("role") == "assistant":
                c = m.get("content", "")
                if isinstance(c, str) and c.strip():
                    return c.strip()
    return ""

def convert_uf(row):
    p = row.get("prompt", "")
    c, r = last_assistant(row.get("chosen")), last_assistant(row.get("rejected"))
    if isinstance(p, str):
        msgs = [{"role": "user", "content": p.strip()}] if p.strip() else []
    elif isinstance(p, list):               # in case the schema is a message list
        msgs = [m for m in p if isinstance(m, dict) and m.get("content")]
    else:
        msgs = []
    if not msgs or msgs[-1].get("role") != "user" or not c or not r or c == r:
        return None
    return {"prompt": to_prompt(msgs), "chosen": c, "rejected": r}

def build(name, ds, convert):
    idx = list(range(len(ds)))
    random.Random(SEED).shuffle(idx)          # deterministic shuffle
    rows, skipped = [], 0
    for i in idx:
        ex = convert(ds[i])
        if ex is None:
            skipped += 1
            continue
        rows.append(ex)
        if len(rows) == TRAIN_SIZE + VAL_SIZE:
            break
    assert len(rows) == TRAIN_SIZE + VAL_SIZE, f"{name}: only {len(rows)} valid pairs"
    val, train = rows[:VAL_SIZE], rows[VAL_SIZE:]
    json.dump(train, open(OUT / f"{name}_train.json", "w"), ensure_ascii=False)
    json.dump(val, open(OUT / f"{name}_validation.json", "w"), ensure_ascii=False)
    print(f"{name}: train={len(train)} val={len(val)} skipped={skipped}")
    return train, val

hh = load_dataset("Anthropic/hh-rlhf", split="train")
uf = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs")
hh_train, hh_val = build("hh_rlhf", hh, convert_hh)
uf_train, uf_val = build("ultrafeedback", uf, convert_uf)

In [ ]:
import numpy as np

for name, rows in [("hh_rlhf", hh_train), ("ultrafeedback", uf_train)]:
    print("=" * 70, "\n", name.upper())
    ex = rows[0]
    print("PROMPT (exactly what the model sees):\n" + ex["prompt"])
    print("CHOSEN  :", repr(ex["chosen"][:200]))
    print("REJECTED:", repr(ex["rejected"][:200]))
    sample = rows[:1000]
    pl = [len(tok(r["prompt"], add_special_tokens=False)["input_ids"]) for r in sample]
    cl = [len(tok(r["chosen"], add_special_tokens=False)["input_ids"]) for r in sample]
    print(f"prompt tokens: median={int(np.median(pl))}  >256: {np.mean(np.array(pl) > 256):.1%}")
    print(f"chosen tokens: median={int(np.median(cl))}")

import os
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(p.relative_to(OUT), f"{p.stat().st_size/1e6:.1f} MB")